#TASK4: Extractive QA
Made by Ángeles Alejandra Cruz Legorreta & Brad Robles García

### 1. Setup and Initialization


In [3]:
!pip install transformers datasets evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


In [4]:
import torch
import numpy as np
import pandas as pd
import time
from datasets import load_dataset
import evaluate
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer
from torch.optim import AdamW

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PyTorch: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


### 2. Dataset Loading and Subsampling


In [5]:
# 1. Load SQuAD v1.1 dataset
squad = load_dataset("rajpurkar/squad")

# 2. Subsample ~15k training examples
# Use a seed for reproducibility
train_subset = squad["train"].shuffle(seed=SEED).select(range(15000))
# Use a small validation subset for faster evaluation during training (e.g., 2000 examples)
validation_subset = squad["validation"].shuffle(seed=SEED).select(range(2000))

print(f"Size of subsampled training set: {len(train_subset)}")
print(f"Size of validation set: {len(validation_subset)}")

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Size of subsampled training set: 15000
Size of validation set: 2000


### 3. Tokenizer and Preprocessing


In [6]:
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

max_length = 384
stride = 128

# Preprocessing function for SQuAD (handling subwords and long context truncation)
def preprocess_training_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

# Apply preprocessing to both Train and Validation sets
train_dataset = train_subset.map(
    preprocess_training_examples,
    batched=True,
    remove_columns=squad["train"].column_names,
)
eval_dataset = validation_subset.map(
    preprocess_training_examples,
    batched=True,
    remove_columns=squad["validation"].column_names,
)
print("Training and validation datasets preprocessed.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Training and validation datasets preprocessed.


### 4. Custom Optimizer (Parameter Groups with 2 Learning Rates)

Requirement: Head wants ~1e-3, pretrained body wants ~2e-5.


In [7]:
def create_optimizer(model):
    # Separate the parameters of the base BERT model and the new QA head
    head_params = []
    base_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "qa_outputs" in name: # This is the head in AutoModelForQuestionAnswering
            head_params.append(param)
        else:
            base_params.append(param)

    # Two parameter groups with different learning rates as per instructions
    optimizer_grouped_parameters = [
        {"params": base_params, "lr": 2e-5},
        {"params": head_params, "lr": 1e-3}
    ]

    return AdamW(optimizer_grouped_parameters)

### 5. Method 1: Partial Fine-Tuning\nFreeze bottom layers, train only top 2 layers + QA Head.


In [8]:
# Load fresh model
model_partial = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

# Freeze all layers first
for param in model_partial.bert.parameters():
    param.requires_grad = False

# Unfreeze the top 2 encoder layers (layer 10 and 11 in BERT-base)
for param in model_partial.bert.encoder.layer[10:].parameters():
    param.requires_grad = True

# The QA head (model_partial.qa_outputs) is automatically unfrozen by default

# Create custom optimizer for Partial Fine-Tuning
optimizer_partial = create_optimizer(model_partial)

training_args_partial = TrainingArguments(
    output_dir="./qa_partial_bert",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=2, # Keep low for Colab efficiency
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=100,
    report_to="none",
    seed=SEED
)

trainer_partial = Trainer(
    model=model_partial,
    args=training_args_partial,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    optimizers=(optimizer_partial, None), # Inject custom optimizer
)

print("Starting Partial Fine-Tuning...")
start_time = time.time()
trainer_partial.train()
time_partial = time.time() - start_time
print(f"Partial Fine-tuning time: {time_partial:.2f} seconds")

results_partial = trainer_partial.evaluate()
print("Partial Fine-Tuning Results:", results_partial)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

Starting Partial Fine-Tuning...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Partial Fine-tuning time: 1060.22 seconds


Training Loss,Validation Loss,Epoch
2.233000,2.135518,2


Partial Fine-Tuning Results: {'eval_loss': 2.1355178356170654}


### 6. Method 2: Full Fine-Tuning\nTrain all layers (Body + Head) simultaneously with two learning rates.


In [9]:
# Load fresh model
model_full = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

# Everything is unfrozen by default in AutoModel

# Create custom optimizer for Full Fine-Tuning
optimizer_full = create_optimizer(model_full)

training_args_full = TrainingArguments(
    output_dir="./qa_full_bert",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=100,
    report_to="none",
    seed=SEED
)

trainer_full = Trainer(
    model=model_full,
    args=training_args_full,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    optimizers=(optimizer_full, None), # Inject custom optimizer
)

print("Starting Full Fine-Tuning...")
start_time = time.time()
trainer_full.train()
time_full = time.time() - start_time
print(f"Full Fine-tuning time: {time_full:.2f} seconds")

results_full = trainer_full.evaluate()
print("Full Fine-Tuning Results:", results_full)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

Starting Full Fine-Tuning...


Epoch,Training Loss,Validation Loss
1,1.425123,1.351217
2,1.032978,1.322171


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Full Fine-tuning time: 2330.56 seconds


Training Loss,Validation Loss,Epoch
1.032978,1.322171,2


Full Fine-Tuning Results: {'eval_loss': 1.3221714496612549}


### 7. Compare Methods


In [10]:
comparison = pd.DataFrame({
    "Method": ["Partial Fine-Tuning (Top 2 + Head)", "Full Fine-Tuning"],
    "Validation Loss": [results_partial["eval_loss"], results_full["eval_loss"]],
    "Training Time (s)": [time_partial, time_full]
})
print(comparison)

                               Method  Validation Loss  Training Time (s)
0  Partial Fine-Tuning (Top 2 + Head)         2.135518        1060.221138
1                    Full Fine-Tuning         1.322171        2330.563583


##MODEL TESTS (The full Fine-tuning one)

In [11]:
contexto_prueba = "Apollo 11 was the first crewed space mission to land on the Moon, on July 20, 1969. Neil Armstrong was the first human to step on the lunar surface."
pregunta_prueba = "Who was the first human to step on the Moon?"

inputs = tokenizer(pregunta_prueba, contexto_prueba, return_tensors="pt")
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = {k: v.to(dispositivo) for k, v in inputs.items()}
model_full.to(dispositivo)
model_full.eval()

with torch.no_grad():
    outputs = model_full(**inputs)
start_idx = torch.argmax(outputs.start_logits, dim=1).item()
end_idx = torch.argmax(outputs.end_logits, dim=1).item()

if start_idx <= end_idx:
    answer_tokens = inputs["input_ids"][0][start_idx:end_idx + 1]
    respuesta = tokenizer.decode(answer_tokens, skip_special_tokens=True)
else:
    respuesta = "Too bad pipipi"

print(f"Context: {contexto_prueba}")
print(f"Question: {pregunta_prueba}")
print(f"Answer: '{respuesta}'")

Context: Apollo 11 was the first crewed space mission to land on the Moon, on July 20, 1969. Neil Armstrong was the first human to step on the lunar surface.
Question: Who was the first human to step on the Moon?
Answer: 'neil armstrong'


In [12]:
contexto_prueba = "The Great Wall of China is an ancient fortification built to protect the borders of the Chinese Empire. Although many believe it is visible from space with the naked eye, astronauts have confirmed that this is just a popular myth."
pregunta_prueba = "Is it possible to see the Great Wall of China from space with the naked eye?"

inputs = tokenizer(pregunta_prueba, contexto_prueba, return_tensors="pt")
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = {k: v.to(dispositivo) for k, v in inputs.items()}
model_full.to(dispositivo)
model_full.eval()

with torch.no_grad():
    outputs = model_full(**inputs)
start_idx = torch.argmax(outputs.start_logits, dim=1).item()
end_idx = torch.argmax(outputs.end_logits, dim=1).item()

if start_idx <= end_idx:
    answer_tokens = inputs["input_ids"][0][start_idx:end_idx + 1]
    respuesta = tokenizer.decode(answer_tokens, skip_special_tokens=True)
else:
    respuesta = "Too bad pipipi"

print(f"Context: {contexto_prueba}")
print(f"Question: {pregunta_prueba}")
print(f"Answer: '{respuesta}'")

Context: The Great Wall of China is an ancient fortification built to protect the borders of the Chinese Empire. Although many believe it is visible from space with the naked eye, astronauts have confirmed that this is just a popular myth.
Question: Is it possible to see the Great Wall of China from space with the naked eye?
Answer: 'visible'


In [13]:
contexto_prueba = "The cheetah is the fastest land animal in the world, capable of reaching speeds of up to 120 kilometers per hour in short sprints. It uses its long, muscular tail to maintain balance and make sharp turns while pursuing prey."
pregunta_prueba = "What is the maximum speed a cheetah can reach in short sprints?"

inputs = tokenizer(pregunta_prueba, contexto_prueba, return_tensors="pt")
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = {k: v.to(dispositivo) for k, v in inputs.items()}
model_full.to(dispositivo)
model_full.eval()

with torch.no_grad():
    outputs = model_full(**inputs)
start_idx = torch.argmax(outputs.start_logits, dim=1).item()
end_idx = torch.argmax(outputs.end_logits, dim=1).item()

if start_idx <= end_idx:
    answer_tokens = inputs["input_ids"][0][start_idx:end_idx + 1]
    respuesta = tokenizer.decode(answer_tokens, skip_special_tokens=True)
else:
    respuesta = "Too bad pipipi"

print(f"Context: {contexto_prueba}")
print(f"Question: {pregunta_prueba}")
print(f"Answer: '{respuesta}'")

Context: The cheetah is the fastest land animal in the world, capable of reaching speeds of up to 120 kilometers per hour in short sprints. It uses its long, muscular tail to maintain balance and make sharp turns while pursuing prey.
Question: What is the maximum speed a cheetah can reach in short sprints?
Answer: '120 kilometers per hour'


In [14]:
contexto_prueba = "The hummingbird is the only bird capable of flying backward and hovering in mid-air thanks to its rapid wingbeats, which can exceed 80 times per second."
pregunta_prueba = "What flying capability makes the hummingbird unique among birds?"

inputs = tokenizer(pregunta_prueba, contexto_prueba, return_tensors="pt")
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = {k: v.to(dispositivo) for k, v in inputs.items()}
model_full.to(dispositivo)
model_full.eval()

with torch.no_grad():
    outputs = model_full(**inputs)
start_idx = torch.argmax(outputs.start_logits, dim=1).item()
end_idx = torch.argmax(outputs.end_logits, dim=1).item()

if start_idx <= end_idx:
    answer_tokens = inputs["input_ids"][0][start_idx:end_idx + 1]
    respuesta = tokenizer.decode(answer_tokens, skip_special_tokens=True)
else:
    respuesta = "Too bad pipipi"

print(f"Context: {contexto_prueba}")
print(f"Question: {pregunta_prueba}")
print(f"Answer: '{respuesta}'")

Context: The hummingbird is the only bird capable of flying backward and hovering in mid-air thanks to its rapid wingbeats, which can exceed 80 times per second.
Question: What flying capability makes the hummingbird unique among birds?
Answer: 'wingbeats'


In [30]:
contexto_prueba = "The Pokémon franchise debuted in 1996 and quickly became a global media empire. Created by Satoshi Tajiri, the core concept was inspired by his childhood hobby of insect collecting, allowing players to catch, train, and trade a variety of creatures. The franchise is famous for its intricate turn-based battle system and expansive lore across over 1,000 species. Each creature possesses unique types, abilities, and sometimes dark biological traits. For example, the Pokédex states that the Psychic-type Spoink must constantly bounce on its tail, as the shock of the bouncing is what keeps its heart beating. The series also continually reinvents itself by introducing regional variants, like Galarian Linoone, and mythical additions like Pecharunt. Beyond video games, the 1997 animated series featuring Ash and Pikachu, along with the Trading Card Game (TCG), turned Pokémon into a cultural touchstone. Later titles like Pokémon GO in 2016 brought these creatures into the real world via augmented reality. Today, Pokémon continues to thrive through main-series updates, bridging the gap between nostalgic older fans and new generations of players."
pregunta_prueba = "What is the name of the Psychic-type pokémon mentioned in the text?"

inputs = tokenizer(pregunta_prueba, contexto_prueba, return_tensors="pt")
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = {k: v.to(dispositivo) for k, v in inputs.items()}
model_full.to(dispositivo)
model_full.eval()

with torch.no_grad():
    outputs = model_full(**inputs)
start_idx = torch.argmax(outputs.start_logits, dim=1).item()
end_idx = torch.argmax(outputs.end_logits, dim=1).item()

if start_idx <= end_idx:
    answer_tokens = inputs["input_ids"][0][start_idx:end_idx + 1]
    respuesta = tokenizer.decode(answer_tokens, skip_special_tokens=True)
else:
    respuesta = "Too bad pipipi"

print(f"Context: {contexto_prueba}")
print(f"Question: {pregunta_prueba}")
print(f"Answer: '{respuesta}'")

Context: The Pokémon franchise debuted in 1996 and quickly became a global media empire. Created by Satoshi Tajiri, the core concept was inspired by his childhood hobby of insect collecting, allowing players to catch, train, and trade a variety of creatures. The franchise is famous for its intricate turn-based battle system and expansive lore across over 1,000 species. Each creature possesses unique types, abilities, and sometimes dark biological traits. For example, the Pokédex states that the Psychic-type Spoink must constantly bounce on its tail, as the shock of the bouncing is what keeps its heart beating. The series also continually reinvents itself by introducing regional variants, like Galarian Linoone, and mythical additions like Pecharunt. Beyond video games, the 1997 animated series featuring Ash and Pikachu, along with the Trading Card Game (TCG), turned Pokémon into a cultural touchstone. Later titles like Pokémon GO in 2016 brought these creatures into the real world via a

##UPLOAD HuggingFace

In [31]:
from huggingface_hub import login, whoami, HfApi
from getpass import getpass
from pathlib import Path

# Iniciar sesión en Hugging Face (Necesitas un Access Token con permisos de "WRITE")
login(token=getpass('HF write token (input hidden): '), add_to_git_credential=False)

repo_id = f"{whoami()['name']}/tds-bert-qa-squad"
private = True

model_dir = Path("./final_qa_model")
model_dir.mkdir(exist_ok=True)
model_full.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)

eval_loss_partial = 2.135518
eval_loss_full = 1.319787

BASE = "bert-base-uncased"
DATASET = "rajpurkar/squad"
winner = "Full Fine-tuning with differentiated learning rates"
EPOCHS = 2
TRAIN_BS = 16
MAX_LEN = 384
SEED = 42

card = f"""---
language: en
pipeline_tag: question-answering
tags:
- bert
- question-answering
- squad
- extractive-qa
metrics:
- loss
model-index:
- name: {repo_id.split('/')[-1]}
  results:
  - task:
      type: question-answering
      name: Extractive Question Answering
    dataset:
      name: SQuAD v1.1
      type: squad
    metrics:
    - name: Validation Loss
      type: loss
      value: {eval_loss_full:.6f}
---
# BERT Extractive QA on SQuAD v1.1 (Winning Model)

## Intended Use
English Extractive Question Answering using the SQuAD v1.1 dataset, developed for an academic course assignment. Not intended for production-critical NLP applications without domain-specific validation.

## Model and Training Details
- **Base Model:** [{BASE}](https://huggingface.co/{BASE})
- **Dataset:** [{DATASET}](https://huggingface.co/datasets/{DATASET}) (Subsampled to 15k training examples).
- **Adaptation Strategy:** {winner}; random seed {SEED}; {EPOCHS} epochs; batch size {TRAIN_BS}.
- **Optimizer Configuration:** Discriminative learning rates (Encoder LR: 2e-5, QA Head LR: 1e-3).
- **Label Alignment:** Used `offset_mapping` to align character-level answer boundaries to WordPiece subtokens.

## Evaluation Results
Evaluation performed on the subsampled validation set (2,000 examples):

- **Validation Loss:** {eval_loss_full:.6f}

### Comparison with Baseline
| Model Variant | Validation Loss |
| :--- | :---: |
| Partial Fine-Tuning (Top 2 layers + Head) | {eval_loss_partial:.6f} |
| **Full Fine-Tuning (This Model)** | **{eval_loss_full:.6f}** |

## Limitations
Trained on a 15k subset of SQuAD v1.1. Performance will degrade on non-English text, informal web text, or questions without an explicit answer in the context (unlike SQuAD v2.0). Sequences exceeding {MAX_LEN} tokens are truncated.

## References
- Rajpurkar et al. (2016), [SQuAD: 100,000+ Questions for Machine Comprehension of Text](https://arxiv.org/abs/1606.05250).
- Devlin et al. (2019), [BERT: Pre-training of Deep Bidirectional Transformers](https://arxiv.org/abs/1810.04805).
- Hugging Face Transformers Documentation.
"""

(model_dir / "README.md").write_text(card, encoding="utf-8")

# 6. Subir a Hugging Face
api = HfApi()
api.create_repo(repo_id=repo_id, private=private, exist_ok=True)
api.upload_folder(repo_id=repo_id, folder_path=str(model_dir), repo_type='model')

print('Published successfully to:', f'https://huggingface.co/{repo_id}')

HF write token (input hidden): ··········


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Published successfully to: https://huggingface.co/BradechiO/tds-bert-qa-squad
